# EXPLANATIONS!

In [ ]:
import logging
import os
import sys
from functools import partial
from logging.handlers import RotatingFileHandler

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from IPython.display import Audio
from jsonschema import validate, ValidationError
from scipy.io.wavfile import write
# from torchcam.methods import GradCAM

import torchaudio
from dog_cat_audio_dataset import CatDogAudioDataset, LABEL_MAP, REVERSE_LABEL_MAP
from bcos.modules.bcosconv2d import BcosConv2d
from config.config_validation_template import CONFIG_TEMPLATE
from custom_logger_formatter import CustomLoggerFormatter
from main import DATASET_MAPPING
from resnet_bcos import make_resnet18
from resnet_18_baseline import BaselineResNet
from tune import TUNABLE_PARAMS
from augment import AudioAugment, SpecAugment

### 0. Config & setup

In [2]:
# EXPERIMENT_DIR = r"output\12-06-2026--22-39\job_0" # change directory to the specific experiment
EXPERIMENT_DIR = os.path.join("output", "12-06-2026--22-39") # change directory to the specific experiment

# BEST_MODEL_DIR = r"best_model" # folder containing .pt model, relative from `EXPERIMENT_DIR`
BEST_MODEL_DIR = os.path.join("job_0", "best_model") # folder containing .pt model, relative from `EXPERIMENT_DIR`

DEVICE = "cuda" # use if CUDA or ROCm

In [3]:
# Initialise Logger.
def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("test logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
# f_ch = RotatingFileHandler(f"{EXPERIMENT_DIR}/explanations.log")
f_ch = RotatingFileHandler(os.path.join(EXPERIMENT_DIR, "explanations.log"))
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the testing process.")

2026-06-13 02:18:54,191 - test logger - INFO - This file logs the testing process. (696224374.py:24)


In [4]:
# validate the provided config file.
# with open(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/run_config.yml", 'r') as stream:
with open(os.path.join(EXPERIMENT_DIR, BEST_MODEL_DIR, "run_config.yml"), 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(os.path.join(EXPERIMENT_DIR, "config.yml"), 'r') as stream:
    general_config = yaml.safe_load(stream)
    print(general_config)
    print(type(general_config))

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

for tunable_param in TUNABLE_PARAMS.keys():
    print(CONFIG[tunable_param])
    if type(CONFIG[tunable_param]) == list:
        CONFIG[tunable_param] = CONFIG[tunable_param][0]

logger.info("config: {CONFIG}")

{'general': {'num_data_workers': 2}, 'jobs': {'job0': {'sample_rate': [22050], 'stride': [5], 'duration': [5], 'n_fft': [1024], 'hop_length': [512], 'n_mels': [128], 'top_db': [60, 100], 'train_val_split': [0.8, 0.2], 'batch_size': [1, 8], 'model': 'resnet18_bcos', 'model_params': {'small_inputs': [0, 1]}, 'optimiser': 'adam', 'b': [2, 5], 'learning_rate': [1e-06, 0.001], 'weight_decay': [1e-06, 0.001], 'n_epochs': 100, 'k_folds': 1, 'tune': True, 'n_trials': 30, 'n_startup_trials': 7, 'min_epochs': 20, 'reduction_factor': 3}}}
<class 'dict'>


KeyError: 'sample_rate'

### 1. Load data


In [ ]:
dataset = CatDogAudioDataset(
    data_dirs=DATASET_MAPPING["train"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Dataset size: {len(dataset)}")
logger.debug(f"Shape of first x element: {dataset[0][0].shape}")
logger.debug(f"First y element: {dataset[0][1]}")
test_data = CatDogAudioDataset(
    data_dirs=DATASET_MAPPING["test"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Test dataset size: {len(test_data)}")

# Normalise
logger.debug("Fitting normalisation.")
dataset.fit_normalisation(list(range(len(dataset))))
test_data.mean = dataset.mean
test_data.std = dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean={dataset.mean}, std={dataset.std}"
)

In [ ]:
# upload your own dataset
# test_data = CatDogAudioDataset(
#     data_dirs="data/personal_recordings/",
#     target_sr=CONFIG["sample_rate"],
#     duration=CONFIG["duration"],
#     n_fft=CONFIG["n_fft"],
#     hop_length=CONFIG["hop_length"],
#     n_mels=CONFIG["n_mels"],
#     top_db=CONFIG["top_db"],
# )
# logger.debug(f"Test dataset size: {len(test_data)}")

# test_data.mean = dataset.mean
# test_data.std = dataset.std

### 2. Load model

In [ ]:
models = {
    "resnet18_bcos": (
        make_resnet18, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1, 
            "small_inputs": True,
            "conv_layer": partial(BcosConv2d, b=2, max_out=2), 
        }
    ),
}
model = None
for name, (cls, kwargs) in models.items():
    if CONFIG['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)


In [ ]:
model_file_path = [entry for entry in os.listdir(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}") if entry.endswith(".pth")][0]

state_dict = torch.load(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/{model_file_path}", weights_only=True)
model.load_state_dict(state_dict)

### 2.5 Get model accuracy

### 3. Create the explanations

In [ ]:
index = 42
SINGLE_TRAIN_IMG = dataset[index][0]
SINGLE_TRAIN_LABEL = dataset[index][1]

plt.imshow(SINGLE_TRAIN_IMG.squeeze(), origin="lower")
plt.title(f"label: {REVERSE_LABEL_MAP[SINGLE_TRAIN_LABEL]}")

display(Audio(dataset.load_waveform(index).squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))
write("test.wav", CONFIG["sample_rate"], dataset.load_waveform(index).squeeze().cpu().numpy())

In [ ]:
model.eval()
# expl_out = model.explain(SINGLE_TRAIN_IMG.unsqueeze(0).to(DEVICE).requires_grad_(), SINGLE_TRAIN_LABEL)
expl_out = model.explain(SINGLE_TRAIN_IMG.unsqueeze(0).to(DEVICE).requires_grad_())

plt.imshow(expl_out["explanation"], origin="lower")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}")
plt.show()

In [ ]:
cm = expl_out["contribution_map"][0].cpu().detach().numpy()

plt.figure(figsize=(8,4))
plt.imshow(cm, origin="lower", cmap="bwr")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}")
plt.colorbar()

In [ ]:
cm = expl_out["contribution_map"][0].cpu().detach().numpy()

logger.info(f"{cm.min() = }, {cm.max() =}")
logger.info(f"{(cm > 0).mean() = }")
logger.info(f"{(cm < 0).mean() = }")

plt.hist(cm.flatten(), bins=100)
plt.show()

In [ ]:
CAT_1, _ = dataset[0]
CAT_2, _ = dataset[3]

DOG_1, _ = dataset[402]
DOG_2, _ = dataset[403]

predict_for_label = "dog"

# --------------------------------------------------
# Build 2x2 grid
#
# CAT | DOG
# ----+----
# CAT | DOG
#
# dog quadrants are top-right and bottom-right
# --------------------------------------------------

def make_grid(a, b, c, d):
    """
    Inputs: tensors (C,H,W)
    Output: tensor (C,2H,2W)
    """

    top = torch.cat([a, b], dim=-1)
    bottom = torch.cat([c, d], dim=-1)

    return torch.cat([top, bottom], dim=-2)

grid_img = make_grid(
    CAT_1,
    DOG_1,
    CAT_2,
    DOG_2,
)

# --------------------------------------------------
# Explain
# --------------------------------------------------

x = (
    grid_img
    .unsqueeze(0)
    .to(DEVICE)
    .requires_grad_()
)

model.eval()

with torch.enable_grad():
    expl_out = model.explain(x, idx=LABEL_MAP[predict_for_label])

cm = expl_out["contribution_map"][0].detach().cpu().numpy()

# --------------------------------------------------
# Grid Pointing Game
# --------------------------------------------------

H, W = cm.shape

peak_y, peak_x = np.unravel_index(
    np.argmax(cm),
    cm.shape
)

# dog quadrants = right half
dog_hit = peak_x >= (W // 2)

logger.info(f"Predicting for: {predict_for_label}")
logger.info(f"Peak location: ({peak_y}, {peak_x})")
logger.info(f"GridPG hit: {dog_hit if predict_for_label == "dog" else not dog_hit}")

# --------------------------------------------------
# Visualisation
# --------------------------------------------------

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# Composite spectrogram
display_img = grid_img.squeeze().cpu().numpy()

ax[0].imshow(display_img, origin="lower")
ax[0].axvline(W//2, color="white")
ax[0].axhline(H//2, color="white")
ax[0].set_title("Composite Grid")

# Contribution map
im = ax[1].imshow(cm, origin="lower", cmap="bwr")
ax[1].scatter(
    peak_x,
    peak_y,
    marker="x",
    s=200,
    c="black"
)
ax[1].axvline(W//2, color="black")
ax[1].axhline(H//2, color="black")
ax[1].set_title("Contribution Map")

plt.colorbar(im, ax=ax[1])

# Positive contributions only
positive_cm = np.maximum(cm, 0)

ax[2].imshow(
    positive_cm,
    origin="lower",
    cmap="hot"
)
ax[2].scatter(
    peak_x,
    peak_y,
    marker="x",
    s=200,
    c="white"
)
ax[2].axvline(W//2, color="white")
ax[2].axhline(H//2, color="white")
ax[2].set_title("Positive Contributions")

plt.tight_layout()
plt.show()

# Convert back to audio

In [ ]:
class STFT:
    def __init__(self):
        self.window = torch.hann_window(CONFIG["n_fft"]).to(DEVICE)

    def forward(self, wav):
        return torch.stft(
            wav,
            n_fft=CONFIG["n_fft"],
            hop_length=CONFIG["hop_length"],
            window=self.window,
            return_complex=True
        )

    def inverse(self, spec):
        return torch.istft(
            spec,
            n_fft=CONFIG["n_fft"],
            hop_length=CONFIG["hop_length"],
            window=self.window
        )
    
def mel_to_stft(unnormalised_log_mel, stft_spec):
    return F.interpolate(
        input=unnormalised_log_mel,
        size=stft_spec.shape,
        mode="bilinear",
        align_corners=False
    )

In [ ]:
model.eval()
# expl_out = model.explain(SINGLE_TRAIN_IMG.unsqueeze(0).to(DEVICE).requires_grad_(), SINGLE_TRAIN_LABEL)
expl_out = model.explain(SINGLE_TRAIN_IMG.unsqueeze(0).to(DEVICE).requires_grad_())

logger.info(expl_out["dynamic_linear_weights"].shape)
logger.info(expl_out["contribution_map"].sum())

linear_weights = expl_out["dynamic_linear_weights"].squeeze(0).squeeze(0).detach().cpu()

plt.imshow(linear_weights, origin="lower")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}", )
plt.colorbar()
plt.show()

In [ ]:
power_mel = torchaudio.functional.DB_to_amplitude(linear_weights, ref=1.0, power=1.0)

plt.imshow(power_mel, origin="lower")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}", )
plt.colorbar()
plt.show()

In [ ]:
normalised_power_mel = (power_mel - power_mel.min()) / (power_mel.max() - power_mel.min() + 1e-8) # normalize to [0, 1]

plt.imshow(normalised_power_mel, origin="lower")
plt.title(f"Prediction: {REVERSE_LABEL_MAP[expl_out["prediction"]]}", )
plt.colorbar()
plt.show()

In [ ]:
def apply_explanation(linear_weights, wav, top_quantile):
    stft = STFT()
    stft_spec = stft.forward(wav).squeeze(0)
    logger.info(f"{stft_spec.shape = }")

    power_mel = torchaudio.functional.DB_to_amplitude(linear_weights, ref=1.0, power=1.0)
    normalised_power_mel = (power_mel - power_mel.min()) / (power_mel.max() - power_mel.min() + 1e-8) # normalize to [0, 1]

    mask = mel_to_stft(normalised_power_mel, stft_spec)
    mask = mask.squeeze(0).squeeze(0)
    
    smoothed_mask = torch.nn.functional.avg_pool2d(
        mask[None, None],
        kernel_size=5,
        stride=1,
        padding=2
    )[0, 0]

    threshold = torch.quantile(smoothed_mask, top_quantile)
    smoothed_mask *= (smoothed_mask >= threshold).float()

    plt.imshow(smoothed_mask.detach().cpu(), origin="lower")
    plt.title(f"Smoothed out mask")
    plt.colorbar()
    plt.show()

    mag = stft_spec.abs() # strength of each time-frequency bin
    mag_weighted = mag * smoothed_mask # Apply weighted top mask (nipple)
    
    phase = stft_spec.angle() # where in the waveform cycle each bin is (important for reconstructing the audio)
    spec_masked = mag_weighted * torch.exp(1j * phase)

    #spec_masked = stft_spec * mask
    wav_out = stft.inverse(spec_masked.unsqueeze(0))

    # print(f"Mask stats - Min: {mask.min()}, Max: {mask.max()}, Mean: {mask.mean()}")
    # print(f"Mean absolute difference: {(mag_masked - mag).abs().mean()}")

    return wav_out

In [ ]:
CAT_IMG = dataset[42][0]
CAT_LABEL = dataset[42][1]
DOG_IMG = dataset[402][0]
DOG_LABEL = dataset[402][1]

x = torch.cat([CAT_IMG, DOG_IMG], dim=-1)
x = x.unsqueeze(0).to(DEVICE).requires_grad_()
x_wav = torch.cat([
    dataset.load_waveform(42),
    dataset.load_waveform(402)
], dim=-1).to(DEVICE)

logger.warning(x.shape)

model.eval()
with torch.enable_grad():
    expl_out_cat = model.explain(x, idx=LABEL_MAP["cat"])
    plt.imshow(expl_out_cat["contribution_map"].detach().squeeze(0).squeeze(0).cpu(), origin="lower")
    plt.title(f"Explain for cat")
    plt.colorbar()
    plt.show()
expl_out_wav_cat = apply_explanation(expl_out_cat["contribution_map"].unsqueeze(0), x_wav.to(DEVICE), 0.9)


with torch.enable_grad():
    expl_out_dog = model.explain(x, idx=LABEL_MAP["dog"])
    plt.imshow(expl_out_dog["contribution_map"].detach().squeeze(0).squeeze(0).cpu(), origin="lower")
    plt.title(f"Explain for dog")
    plt.colorbar()
    plt.show()
expl_out_wav_dog = apply_explanation(expl_out_dog["contribution_map"].unsqueeze(0), x_wav.to(DEVICE), 0.9)

print("Original CAT + DOG audio:")
display(Audio(x_wav.squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))

print("Explanation for CAT prediction:")
display(Audio(expl_out_wav_cat.detach().squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))

print("Explanation for DOG prediction:")
display(Audio(expl_out_wav_dog.detach().squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))

# --------------------------------------------------
# Storage of the masked audio for sharing
# --------------------------------------------------

write("original_DOG_CAT.wav", CONFIG["sample_rate"], x_wav.squeeze().cpu().numpy())
write("explanation_CAT.wav", CONFIG["sample_rate"], expl_out_wav_cat.detach().squeeze().cpu().numpy())
write("explanation_DOG.wav", CONFIG["sample_rate"], expl_out_wav_dog.detach().squeeze().cpu().numpy())


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

ax[0].imshow(expl_out_cat["explanation"], origin="lower", cmap="magma")
xmin, xmax = ax[0].get_xlim()
center = (xmin + xmax) / 2
ax[0].axvline(center, color="white")
ax[0].set_title(f"Prediction: cat")

ax[1].imshow(expl_out_dog["explanation"], origin="lower", cmap="magma")
ax[1].axvline(center, color="white")
ax[1].set_title(f"Prediction: dog")

plt.tight_layout()
plt.show()


In [ ]:
explanation_cat = expl_out_cat["explanation"].squeeze() * x.detach().cpu().numpy().squeeze()
explanation_cat = (explanation_cat - explanation_cat.min()) / (explanation_cat.max() - explanation_cat.min() + 1e-8) # normalize to [0, 1]

plt.imshow(explanation_cat, origin="lower")
plt.title(f"Prediction: cat")
plt.show()


In [ ]:
explanation_cat = expl_out_cat["contribution_map"].detach().cpu().squeeze()# * x.detach().cpu().numpy().squeeze()
explanation_cat = (explanation_cat - explanation_cat.min()) / (explanation_cat.max() - explanation_cat.min() + 1e-8) # normalize to [0, 1]

plt.imshow(explanation_cat, origin="lower")
plt.title(f"Prediction: cat")
plt.show()


### Baseline ResNet Explanations

### 0. Config & setup


In [ ]:
EXPERIMENT_DIR = r"output\11-06-2026--10-27" # change directory to the specific experiment

BEST_MODEL_DIR_RN = r"job_1" # folder containing .pt model, relative from `EXPERIMENT_DIR`

DEVICE = "cuda" # use if CUDA or ROCm

In [ ]:
# validate the provided config file.
with open(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR_RN}/run_config.yml", 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(f"{EXPERIMENT_DIR}/config.yml", 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

for tunable_param in TUNABLE_PARAMS.keys():
    if type(CONFIG[tunable_param]) == list:
        CONFIG[tunable_param] = CONFIG[tunable_param][0]

logger.info("config: {CONFIG}")

### Load Model

In [ ]:
models_baseline = {
    "resnet18_baseline": (
            lambda **kwargs: BaselineResNet(kwargs["num_classes"]),
            {"logger": logger, "num_classes": dataset.get_n_classes()}
    )   
    }
model_bl = None
for name, (cls, kwargs) in models_baseline.items():
    if CONFIG['model'].lower() in name:
        model_bl = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model_bl})."

logger.debug(f"Model:\n{model_bl}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model_bl = model_bl.to(DEVICE)

In [ ]:
model_file_path = [entry for entry in os.listdir(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR_RN}") if entry.endswith(".pth")][0]

state_dict = torch.load(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR_RN}/{model_file_path}", weights_only=True)
model_bl.load_state_dict(state_dict)

### Initializing GradCAM

In [ ]:
# Load baseline model
num_classes = dataset.get_n_classes()
model_baseline = BaselineResNet(num_classes=num_classes).to(DEVICE)

checkpoint_folder = os.path.join(EXPERIMENT_DIR, BEST_MODEL_DIR_RN)

checkpoint_path = [
    os.path.join(checkpoint_folder, f)
    for f in os.listdir(checkpoint_folder)
    if f.endswith(".pth")
][0]

state_dict = torch.load(checkpoint_path, weights_only=True)

model_baseline.load_state_dict(state_dict)
model_baseline.eval()

# Select example
index = 5
x = dataset[index][0].unsqueeze(0).to(DEVICE)
x_vis = x.squeeze().detach().cpu().numpy()
x_vis = (x_vis - x_vis.min()) / (x_vis.max() - x_vis.min() + 1e-8)

# Grad-CAM
cam_extractor = GradCAM(model_baseline.model, target_layer="layer4")
y_hat = model_baseline.model(x)
pred_class = y_hat.argmax(dim=1).item()

cams = cam_extractor(pred_class, y_hat)[0]
cams_resized = F.interpolate(
    cams.unsqueeze(0),
    size=x.shape[-2:],
    mode="bilinear",
    align_corners=False
)

# Plot
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(x_vis, origin="lower", cmap="magma")
plt.title(f"Original Spectrogram (True: {REVERSE_LABEL_MAP[dataset[index][1]]})")
plt.xlabel("Time")
plt.ylabel("Mel Bin")
plt.colorbar(label="Normalized amplitude")

plt.subplot(1, 2, 2)
plt.imshow(x_vis, origin="lower", cmap="magma")
plt.imshow(
    cams_resized.squeeze().cpu().detach().numpy(),
    origin="lower",
    cmap="jet",
    alpha=0.5
)
plt.title(f"Grad-CAM Overlay (Pred: {REVERSE_LABEL_MAP[pred_class]})")
plt.xlabel("Time")
plt.ylabel("Mel Bin")
plt.colorbar(label="Grad-CAM intensity")

plt.tight_layout()
plt.show()

### Audio masking with Grad-CAM explanations for baseline ResNet

In [ ]:
# Load baseline model using the current job folder
num_classes = dataset.get_n_classes()
model_baseline = BaselineResNet(num_classes=num_classes).to(DEVICE)

checkpoint_folder = os.path.join(EXPERIMENT_DIR, BEST_MODEL_DIR_RN)

checkpoint_path = [
    os.path.join(checkpoint_folder, f)
    for f in os.listdir(checkpoint_folder)
    if f.endswith(".pth")
][0]

state_dict = torch.load(checkpoint_path, weights_only=True)
model_baseline.load_state_dict(state_dict)
model_baseline.eval()

print(f"Loaded baseline checkpoint from: {checkpoint_path}")

CAT_IDX = 4
DOG_IDX = 407

CAT_IMG = dataset[CAT_IDX][0]
DOG_IMG = dataset[DOG_IDX][0]

x = torch.cat([CAT_IMG, DOG_IMG], dim=-1).unsqueeze(0).to(DEVICE)
x_wav = torch.cat(
    [dataset.load_waveform(CAT_IDX), dataset.load_waveform(DOG_IDX)],
    dim=-1
).to(DEVICE)


x_vis = x.squeeze().detach().cpu().numpy()
x_vis = (x_vis - x_vis.min()) / (x_vis.max() - x_vis.min() + 1e-8)

cam_extractor = GradCAM(model_baseline.model, target_layer="layer4")

cat_idx = LABEL_MAP["cat"]
y_hat_cat = model_baseline.model(x)
cams_cat = cam_extractor(cat_idx, y_hat_cat)[0]
cams_cat_resized = F.interpolate(
    cams_cat.unsqueeze(0),
    size=x.shape[-2:],
    mode="bilinear",
    align_corners=False
)
cams_cat_resized = (
    cams_cat_resized - cams_cat_resized.min()
) / (cams_cat_resized.max() - cams_cat_resized.min() + 1e-8)

dog_idx = LABEL_MAP["dog"]
y_hat_dog = model_baseline.model(x)
cams_dog = cam_extractor(dog_idx, y_hat_dog)[0]
cams_dog_resized = F.interpolate(
    cams_dog.unsqueeze(0),
    size=x.shape[-2:],
    mode="bilinear",
    align_corners=False
)
cams_dog_resized = (
    cams_dog_resized - cams_dog_resized.min()
) / (cams_dog_resized.max() - cams_dog_resized.min() + 1e-8)


def apply_gradcam_mask(cam_map, wav):
    stft = STFT()
    stft_spec = stft.forward(wav).squeeze(0)

    cam_2d = cam_map.squeeze(0).squeeze(0)

    cam_resized = F.interpolate(
        cam_2d.unsqueeze(0).unsqueeze(0),
        size=stft_spec.shape,
        mode="bilinear",
        align_corners=False
    ).squeeze(0).squeeze(0)

    mask = (cam_resized - cam_resized.min()) / (
        cam_resized.max() - cam_resized.min() + 1e-8
    )
    mask = mask ** 2.5

    mag = stft_spec.abs()
    phase = stft_spec.angle()

    threshold = torch.quantile(mask, 0.9)
    binary_mask = (mask >= threshold).float()

    mag_weighted = mag * binary_mask
    alpha = 3
    mag_masked = mag_weighted * (1.0 + alpha * mask)
    spec_masked = mag_masked * torch.exp(1j * phase)

    wav_out = stft.inverse(spec_masked.unsqueeze(0))
    return wav_out


expl_out_wav_cat = apply_gradcam_mask(cams_cat_resized, x_wav)
expl_out_wav_dog = apply_gradcam_mask(cams_dog_resized, x_wav)

write("original_DOG_CAT.wav", CONFIG["sample_rate"], x_wav.squeeze().cpu().numpy())
write("explanation_CAT_gradcam.wav", CONFIG["sample_rate"], expl_out_wav_cat.squeeze().cpu().numpy())
write("explanation_DOG_gradcam.wav", CONFIG["sample_rate"], expl_out_wav_dog.squeeze().cpu().numpy())

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(x_vis, origin="lower", aspect="auto", cmap="magma")
plt.imshow(
    cams_cat_resized.squeeze().cpu().detach().numpy(),
    origin="lower",
    aspect="auto",
    cmap="jet",
    alpha=0.5
)
plt.title("Grad-CAM Overlay (Cat prediction)")
plt.xlabel("Time")
plt.ylabel("Mel Bin")
plt.colorbar(label="Importance")

plt.subplot(1, 2, 2)
plt.imshow(x_vis, origin="lower", aspect="auto", cmap="magma")
plt.imshow(
    cams_dog_resized.squeeze().cpu().detach().numpy(),
    origin="lower",
    aspect="auto",
    cmap="jet",
    alpha=0.5
)
plt.title("Grad-CAM Overlay (Dog prediction)")
plt.xlabel("Time")
plt.ylabel("Mel Bin")
plt.colorbar(label="Importance")

plt.tight_layout()
plt.show()

print("Original CAT + DOG audio:")
display(Audio(x_wav.squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))

print("Grad-CAM explanation for CAT:")
display(Audio(expl_out_wav_cat.squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))

print("Grad-CAM explanation for DOG:")
display(Audio(expl_out_wav_dog.squeeze().cpu().numpy(), rate=CONFIG["sample_rate"]))